# Output Parser
**Output Parser**는 대규모 언어 모델(LLM, Large Language Model)의 출력 결과를 **애플리케이션에서 활용할 수 있도록 적절한 형식으로 변환**하는 도구이다.
- LLM은 일반적으로 텍스트 형태로 응답을 생성하지만, 이 텍스트는 그대로 활용하기 어려운 경우가 많다.
- Output Parser는 이러한 **비구조적 텍스트 데이터를 구조화된 데이터로 변환**하여 프로그램에서 활용 가능하도록 만든다.
- 예를 들어, 키워드 리스트를 뽑거나 JSON 형식으로 정보를 변환하는 데 사용된다.

## 주요 Output Parser 종류

1. **CommaSeparatedListOutputParser**
   - 쉼표로 구분된 텍스트를 파싱하여 리스트 형태로 변환한다.
   - 예: `"사과, 바나나, 포도"` → `["사과", "바나나", "포도"]`
2. **JsonOutputParser**
   - LLM의 출력이 JSON 형식일 때 이를 Python의 `dict` 객체로 변환한다.
   - JSON(JavaScript Object Notation)은 데이터 구조를 표현하기 위한 경량 포맷이다.
3. **PydanticOutputParser**
   - JSON 데이터를 Python의 [Pydantic](https://docs.pydantic.dev) 모델로 변환한다.
   - Pydantic은 데이터 유효성 검사와 설정 관리에 널리 사용되는 Python 라이브러리이다.
4. **StrOutputParser**
   - 모델의 출력 결과를 단순 문자열로 반환한다.(응답 문자열만 return. ex. print(res.content))
   - Chat 기반 모델은 Message 객체의 속성으로 LLM 결과를 반환한다. 거기에서 응답 문자열만 추출해서 반환한다.
> `JsonOutputParser`, `PydanticOutputParser` 는 모두 Pydantic을 사용해 데이터 구조(schema)를 정의하고, 해당 구조에 따라 출력을 검증하고 변환한다.

## 주요 메소드
- `parse(text: str)`
  - LLM이 생성한 문자열 응답을 받아 '정해진 구조로 변환'하여 반환한다.
- `get_format_instructions() -> str`
  - 각 OutputParer가 '변환할 수있는 형식으로 LLM이 응답하도록 하는 프롬프트 텍스트'를 반환한다.(응답 포맷에 대한 지침)
  - 이 내용을 프롬프트에 넣어서 LLM이 정확한 포맷으로 응답하도록 유도한다.
  
## 참고
- Output Parser는 일반적으로 [`Runnable`](05_chaing_LECL.ipynb#Runnable) 인터페이스를 상속하여 구현되며, `invoke()` 메서드를 통해 실행할 수 있다.
- `invoke()`는 내부적으로 `parse()`를 호출하여 동작한다.
- 필요한 경우 Output Parser를 직접 구현하여 사용자 정의 출력 포맷을 처리할 수도 있다. 


## StrOutputParser
- 모델(LLM)의 출력 결과를 string으로 변환하여 반환하는 output parser.
- Chat Model은  Message 객체에서 content 속성값을 추출하여 문자열로 반환한다.

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()


True

In [6]:
prompt = ChatPromptTemplate.from_template(
    template="한국의 {topic}에 관련된 속담 {count}개를 알려줘. 네가 만들지 말고 실제 있는 속담으로, 출력은 목록 형식으로 해줘."
)
model = ChatOpenAI(model="gpt-5.4-nano")
parser = StrOutputParser()

# prompt --[Query]--> model --[response:AIMeassage]--> parser --[문자열]--> 최종 결과
query = prompt.invoke({"topic":"호랑이", "count":3})
print(query)
response = model.invoke(query)
response

messages=[HumanMessage(content='한국의 호랑이에 관련된 속담 3개를 알려줘. 네가 만들지 말고 실제 있는 속담으로, 출력은 목록 형식으로 해줘.', additional_kwargs={}, response_metadata={})]


AIMessage(content='- **호랑이에게 물려가도 정신만 차리면 산다**  \n- **굴러온 돌이 호랑이 새끼보다 낫다**  \n- **범은 범을 낳고, 호랑이는 호랑이를 낳는다**', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 43, 'total_tokens': 103, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Dto21Op77CmyZmW0eZmFzAQ5WJqg8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ef2f6-fadd-7d13-8688-76bf249da263-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 60, 'total_tokens': 103, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
res_str = parser.invoke(response)
print(res_str)
print(response.content)

- **호랑이에게 물려가도 정신만 차리면 산다**  
- **굴러온 돌이 호랑이 새끼보다 낫다**  
- **범은 범을 낳고, 호랑이는 호랑이를 낳는다**
- **호랑이에게 물려가도 정신만 차리면 산다**  
- **굴러온 돌이 호랑이 새끼보다 낫다**  
- **범은 범을 낳고, 호랑이는 호랑이를 낳는다**


## CommaSeparatedListOutputParser

- 쉼표로 구분된 텍스트를 파싱하여 리스트 형태로 변환한다.
  - "a,b,c" => ['a','b','c']

In [13]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser = CommaSeparatedListOutputParser()
txt = "서울, 인천, 부산, 광주, 대구, 대전, 울산"
# txt = "찾은 도시 이름은 서울과 인천과 대전입니다."

r1 = parser.parse(txt)
r2 = parser.invoke(txt)

print(r1)
print(r2)


['서울', '인천', '부산', '광주', '대구', '대전', '울산']
['서울', '인천', '부산', '광주', '대구', '대전', '울산']


In [14]:
# LLM 모델이 Output Parser의 맞는 출력을 하도록 프롬프트에 넣을 지침 조회
instruction = parser.get_format_instructions()
print(instruction)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [16]:
prompt = ChatPromptTemplate(
    messages=[
        {"role":"system", "content":"출력 형식 : {output_format}"},
        {"role":"user", "content":"{query}"}
    ],
    partial_variables={"output_format":instruction} # input_variables에 값을 프롬프트 템플릿 생성하면서 입력
) 
model = ChatOpenAI(model="gpt-5.4-nano")

query = prompt.invoke({"query":"자동차 종류 다섯 가지를 알려줘."})
response = model.invoke(query)

In [22]:
response.content


'세단, SUV, 해치백, 쿠페, 픽업트럭'

In [21]:
res = parser.invoke(response)
print(res)

['세단', 'SUV', '해치백', '쿠페', '픽업트럭']


## JsonOutputParser

- JSON(JavaScript Object Notation) 형식의 응답을 dictionary로 반환한다.
- JSON 형식을 정하려는 경우 [Pydantic](Ref_typing_Pydantic.ipynb)을 이용해 JSON 스키마를 정의하여 JsonOutputParser 생성시 전달한다.
  - Pydantic 모델클래스를 이용해 LLM 모델이 응답할 때 json의 어떤 key에 어떤 응답을 작성할 지 Field로 정의한다.
  - Schema 지정은 필수는 아니다. 
- LLM이 JSON Schema를 따르는 형태로 응답을 하면 JsonOutputParser는 Dictionary로 변환한다.

In [26]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
# LLM 출력 형식 프롬포트
print(parser.get_format_instructions())

txt = '{"name":"이순신", "age":"30", "address":"서울"}'
r1 = parser.parse(txt)
r2 = parser.invoke(txt)

print(type(r1), r1)
print(type(r2), r2)

Return a JSON object.
<class 'dict'> {'name': '이순신', 'age': '30', 'address': '서울'}
<class 'dict'> {'name': '이순신', 'age': '30', 'address': '서울'}


In [29]:
parser = JsonOutputParser()
prompt = ChatPromptTemplate(
    messages=[
        ("system", "output format: {output_format}"),
        ("user", "{query}")
    ],
    partial_variables={"output_format":parser.get_format_instructions()}
)
model = ChatOpenAI(model="gpt-5.4-mini")

query = prompt.invoke({"query":"이순신 장군에 대해서 알려줘."})
# query.messages
res = model.invoke(query)

In [36]:
print(res.content)
# final_answer = parser.invoke(res)

{"answer":"이순신(李舜臣, 1545~1598)은 조선 중기의 대표적인 장군으로, 임진왜란 때 조선 수군을 이끌고 일본군을 상대로 큰 승리를 거둔 인물입니다. 한국 역사에서 가장 존경받는 장군 중 한 명으로 꼽힙니다.\n\n핵심만 정리하면:\n- 직책: 조선의 무신, 수군 절도사\n- 대표 업적: 한산도 대첩, 명량 해전, 노량 해전 등에서 승리\n- 특징: 뛰어난 전략가, 강한 책임감, 청렴한 성품\n- 최후: 1598년 노량 해전에서 전사\n\n주요 이야기:\n1. 임진왜란이 일어나자 조선 수군을 정비해 일본군의 해상 보급로를 차단했습니다.\n2. 거북선을 활용해 해전에서 큰 전과를 올렸습니다.\n3. 특히 명량 해전에서는 매우 불리한 조건에서도 대승을 거두었습니다.\n4. 마지막 전투인 노량 해전에서 전사했으며, \"싸움이 한창이니 나의 죽음을 알리지 말라\"는 말로 유명합니다.\n\n이순신 장군은 단순히 전쟁에서 이긴 장군이 아니라, 나라가 가장 어려울 때 끝까지 책임을 다한 인물로 기억됩니다.\n\n원하시면 제가 다음 중 하나로 더 자세히 설명해드릴 수 있어요:\n- 생애를 연표처럼 정리\n- 임진왜란에서의 활약 자세히 설명\n- 거북선에 대해 설명\n- 이순신의 리더십과 명언 정리"}


In [ ]:
# final_answer['answer']
# final_answer.keys()

dict_keys(['answer'])

In [37]:
# JSON 형식에 대한 스키마 설계
## 어떤 키를 가지며, 그 키에 어떤 값을 넣어야 하는지 설계 => Pydantic 모델 이용
from pydantic import BaseModel, Field
class PersonInfoSchema(BaseModel):
    """
    인물 정보에 대한 응답을 위한 Schema
    """
    # 변수명(key): 결과 값의 타입 = Field(description="어떤 값을 넣을지 설명")
    name: str = Field(description="조회한 사람의 이름")
    yob: int = Field(description="조회한 사람의 출생 년도. 모를 경우는 -1을 대입")
    yod: int = Field(description="조회한 사람의 사망 년도. 모를 경우는 -1을 대입")
    profile: str = Field(description="조회한 사람의 주요 업적 소개")

parser2 = JsonOutputParser(pydantic_object=PersonInfoSchema)
# parser.get_format_instructions()
print(parser2.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [42]:
prompt2 = ChatPromptTemplate(
    messages=[
        ("system", "output format: {output_format}"),
        ("user", "{query}")
    ],
    partial_variables={"output_format":parser2.get_format_instructions()}
)

query2 = prompt2.invoke({"query":"고구려의 온달장군에 대해 알려줘."})
res2 = model.invoke(query2)

In [44]:
# res2.content
answer = parser2.invoke(res2)
answer
# answer['name']

{'name': '온달',
 'yob': -1,
 'yod': -1,
 'profile': "고구려의 장군으로 널리 알려진 인물입니다. 전설과 설화에서 평강공주와의 이야기로 유명하며, 『삼국사기』에 따르면 평원왕 때 말갈과의 전투에서 전사한 인물로 전해집니다. '바보 온달' 이야기로 민중에게 친숙하지만, 역사적으로는 용맹한 무장으로 기억됩니다."}

## PydanticOutputParser

- JSON 형태로 받은 응답을 Pydantic 모델로 변환하여 반환한다.
- 구현은 JsonOutputParser와 동일한데 parsing 결과를 pydantic 모델타입으로 반환한다.

In [1]:
from pydantic import BaseModel, Field
class PersonInfoSchema(BaseModel):
    """
    인물 정보에 대한 응답을 위한 Schema
    """
    # 변수명(key): 결과 값의 타입 = Field(description="어떤 값을 넣을지 설명")
    name: str = Field(description="조회한 사람의 이름")
    yob: int = Field(description="조회한 사람의 출생 년도. 모를 경우는 -1을 대입")
    yod: int = Field(description="조회한 사람의 사망 년도. 모를 경우는 -1을 대입")
    profile: str = Field(description="조회한 사람의 주요 업적 소개")

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI

parser = PydanticOutputParser(pydantic_object=PersonInfoSchema)
# print(parser.get_format_instructions())
prompt = ChatPromptTemplate(
    messages=[
        ("system", "당신은 초등학교 역사 선생님입니다. 다음 출력  형식에 맞춰 답변해주세요.\n\n<출력형식>{output_format}</출력형식>"),
        ("user", "{query}")
    ],
    partial_variables = {"output_format":parser.get_format_instructions()}
)
model = ChatOpenAI(model="gpt-5.4-mini")

In [9]:
# 1. Prompt 생성
query = prompt.invoke({"query":"세종대왕에 대해서 알려주세요."})
# print(query)

# 2. model에게 요청
res = model.invoke(query)
print(type(res), res)

# 3. 응답을 응답 format에 맞게 parsing
final_answer = parser.invoke(res)

<class 'langchain_core.messages.ai.AIMessage'> content='{"name":"세종대왕","yob":1397,"yod":1450,"profile":"조선의 4대 왕으로, 한글을 창제하여 백성들이 쉽게 글을 읽고 쓸 수 있게 했습니다. 과학, 농업, 음악, 천문 등 여러 분야를 크게 발전시킨 훌륭한 임금입니다."}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 342, 'total_tokens': 425, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-Du5xbaAwASsEpybYLqW83pLfnqBvm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ef712-9619-7912-b0fe-dcc31f093b9d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 342, 'output_tokens': 83, 'total_tokens': 425, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details

In [12]:
print(type(final_answer))
print(final_answer)

<class '__main__.PersonInfoSchema'>
name='세종대왕' yob=1397 yod=1450 profile='조선의 4대 왕으로, 한글을 창제하여 백성들이 쉽게 글을 읽고 쓸 수 있게 했습니다. 과학, 농업, 음악, 천문 등 여러 분야를 크게 발전시킨 훌륭한 임금입니다.'


In [14]:
print(final_answer.name)
print(final_answer.yob, final_answer.yod)
print(final_answer.profile)

세종대왕
1397 1450
조선의 4대 왕으로, 한글을 창제하여 백성들이 쉽게 글을 읽고 쓸 수 있게 했습니다. 과학, 농업, 음악, 천문 등 여러 분야를 크게 발전시킨 훌륭한 임금입니다.


# LLM모델에 출력 형식을 설정

- ChatModel객체의 `with_structured_output(pydantic.BaseModel)` 을 이용해 모델의 출력 형식을 모델 자체에 추가할 수있다.
- `OutputParser`는 모델의 출력 결과를 받아서 형식을 변경해 준다. 그래서 Chain에 탈/부착을 통해 형식을 적용하거나 적용하지 않는 것을 자유롭게 할 수있다.
- 모델의 출력 결과를 항상 일정하게 할 경우에는 아예 **모델에 출력 형식을 설정할 수 있다.**

In [ ]:
model_with_output = model.with_structured_output(PersonInfoSchema)
res = model_with_output.invoke("세종대왕에 대해서 설명해주세요.") # 출력 형식이 설정된 모델 호출

# model.invoke("세종대왕에 대해서 설명해주세요.") # 출력 형식이 설정 안 된 모델 호출

In [ ]:
res
# res.name

'세종대왕'

In [20]:
res = model_with_output.invoke("AI Agent의 특정과 장단점에 대해서 정리해서 설명해줘")

In [21]:
res

PersonInfoSchema(name='AI Agent', yob=-1, yod=-1, profile='AI Agent(인공지능 에이전트)는 사용자의 목표를 이해하고, 환경을 인식하며, 계획을 세워 행동을 수행하는 소프트웨어 또는 시스템입니다. 단순히 질문에 답하는 챗봇과 달리, 여러 단계를 거쳐 작업을 진행하고 외부 도구(API, 검색, 데이터베이스, 자동화 툴 등)를 활용해 실제 업무를 수행할 수 있습니다.\n\n특징:\n- 목표 지향적: 사용자의 의도를 바탕으로 목표를 달성하려고 함\n- 자율성: 일정 범위 내에서 스스로 판단하고 행동함\n- 도구 활용: 검색, 계산, 파일 처리, 예약, 코드 실행 등과 연동 가능\n- 반복 수행: 계획-실행-검토 과정을 반복하며 결과를 개선함\n- 상황 인식: 이전 대화, 상태, 외부 정보를 고려해 행동함\n\n장점:\n- 반복적 업무 자동화로 생산성 향상\n- 여러 단계의 복합 작업을 빠르게 처리 가능\n- 24시간 동작 가능\n- 사람의 실수를 줄이고 일관성 있는 결과를 제공\n- 개인화된 지원과 의사결정 보조에 유리함\n\n단점:\n- 잘못된 판단이나 환각(hallucination) 가능성\n- 보안/개인정보 유출 위험\n- 초기 구축 및 운영 비용이 높을 수 있음\n- 복잡한 상황에서 예측 불가능한 행동을 할 수 있음\n- 책임 소재가 불분명해질 수 있음\n\n활용 예:\n- 고객응대 자동화\n- 일정 예약 및 업무 보조\n- 데이터 분석과 리포트 생성\n- 마케팅 자동화\n- 개발 지원 및 코드 보조')

# Streaming 방식 응답 처리

- Streaming 방식 응답 처리란, LLM이 텍스트를 모두 생성할 때까지 기다리지 않고, 생성되는 즉시 **부분적인 결과**를 실시간으로 전달받아 처리하는 방식을 의미한다. 이는 사용자가 응답을 더 빠르게 인지할 수 있게 해 주며, 특히 대화형 서비스, 실시간 UI 출력, 긴 문서 생성과 같은 상황에서 매우 유용하게 활용된다.

- `invoke()` 요청으로 받는 응답은 **비 스트리밍 방식**으로 모든 응답 텍스트 생성이 완료된 이후 그 결과를 한 번에 반환하는 구조이다. 반면 Streaming 방식은 **토큰(token) 단위 또는 여러 토큰이 묶인 청크(chunk) 단위**로 연속적인 데이터 스트림을 전송한다는 점에서 큰 차이가 있다. 즉, Streaming은 마치 사람이 타이핑을 치듯이 응답을 실시간으로 “흘려보내는” 방식이라고 이해할 수 있다.

- `모델.invoke(input, config)` → 응답 데이터
    - 모델이 전체 응답을 모두 생성한 뒤, 최종 결과를 한 번에 반환하는 방식이다.
    - 배치 처리나 후처리가 중요한 경우에 적합하다.
- `모델.stream(input, config)` → generator
    - 모델이 토큰을 생성하는 즉시, 순차적으로 제공하는 generator 를 반환한다.
    - 실시간 출력, 대화형 인터페이스, 웹 스트리밍 등에 특히 적합하다.

In [22]:
model = ChatOpenAI(model="gpt-5.4-nano")
res = model.invoke("서울에 유명한 여행지 세 곳을 소개해 주세요. 간단한 소개도 해 주세요.")
print(res.content)

물론입니다! 서울에서 사람들이 많이 찾는 **유명한 여행지 3곳**을 간단히 소개해드릴게요.

1) **경복궁 (Gyeongbokgung Palace)**
- 조선 왕조의 대표적인 궁궐로, 웅장한 건축과 아름다운 전통 분위기를 느낄 수 있어요.  
- 시간대에 따라 **수문장 교대의식** 같은 볼거리도 즐길 수 있습니다.

2) **명동 거리 (Myeongdong)**
- 쇼핑과 길거리 음식으로 유명한 서울의 대표적인 번화가예요.  
- 로드샵, 카페, 먹거리(김밥/떡볶이/야식 등)가 많아 관광하면서 재미있게 둘러보기 좋습니다.

3) **남산타워 (N서울타워)**
- 남산 정상에서 서울 전경을 한눈에 볼 수 있어요.  
- 특히 밤에는 야경이 정말 아름답고, 데이트 코스로도 인기가 많습니다.

원하시면 **여행 일정(예: 1박2일/2박3일)** 형태로 묶어서 동선까지 추천해드릴까요?


In [25]:
gen = model.stream("서울에 유명한 여행지 세 곳을 소개해 주세요. 간단한 소개도 해 주세요.")
print(gen)

<generator object BaseChatModel.stream at 0x000002A25EBF6CD0>


In [26]:
for token in gen:
    print(token.content, end="")

물론이에요! 서울에서 유명한 여행지 3곳을 간단히 소개해 드릴게요.

1) **경복궁**
- 조선 왕조의 대표적인 궁궐로, 한복을 입고 관람하기에도 좋아요. 아름다운 전각과 역사적인 분위기를 느낄 수 있습니다.  

2) **명동(명동거리)**
- 쇼핑과 먹거리로 유명한 서울의 대표적인 번화가예요. 길거리 음식부터 다양한 브랜드 매장까지 즐길 수 있어요.  

3) **남산서울타워(남산공원)**
- 도심 전경을 한눈에 내려다볼 수 있는 전망 명소입니다. 특히 저녁에 야경을 보기 좋아서 사진 찍기에도 인기예요.  

원하시면 “가족용/커플용/혼자 여행용”처럼 취향에 맞춰서 추천도 해드릴까요?